## 오라클 연동

- 보통 `Oracledb` 라이브러리 사용. 이전에는 `cx_Oracle`이라고 사용했음
- pip로 Oracledb 설치

    ```bash
    pip install oracledb
    ```

In [37]:
# 설치
!pip install oracledb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Oracle 연결 시작

- oracledb 라이브러리 소스에 추가
- 접속정보를 입력
- SQL쿼리 실행 확인 

#### DB 연결시 필요정보
- DB 접속정보 (DataSource Namespace)
- 연결개체 (Connection)
- 커서 (Cursor)
- 트랜잭션 (Commit, Rollback)

#### SELECT 실행

In [38]:
import oracledb

In [39]:
oracledb.__version__

'3.4.2'

In [40]:
# DB접속정보 설정
## jdbc:oracle:thin:@localhost:1521/XE 중에서 localhost이후 부터 사용
dsn = 'localhost:1521/XE'

conn = oracledb.connect(
    user = 'test',
    password = 'test12345',
    dsn = dsn
)

print('Oracle 연결 성공!', conn.version)

Oracle 연결 성공! 21.3.0.0.0


In [41]:
# 커서 생성
cursor = conn.cursor()

# SQL문 실행
sql = """
SELECT dept_id, dept_name, loc
  FROM dept
 ORDER BY dept_id
"""

# 실행 
cursor.execute(sql)

# 결과출력
for row in cursor:
    print(row)

(10, '개발부', 'BUSAN')
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
(40, '인사부', 'BUSAN')
(100, '생산부', 'KWANGJU')


In [42]:
# 종료
cursor.close()
conn.close()

#### 일반적인 방법

- 접속 후 
- 비지니스 트랜잭션 처리가 끝나면
- 접속 종료

In [43]:
import oracledb

# DB접속정보 설정
## jdbc:oracle:thin:@localhost:1521/XE 중에서 localhost이후 부터 사용
dsn = 'localhost:1521/XE'

conn = oracledb.connect(
    user = 'test',
    password = 'test12345',
    dsn = dsn
)

print('Oracle 연결 성공!', conn.version)

# 커서 생성
cursor = conn.cursor()

# SQL문 실행
sql = """
SELECT dept_id, dept_name, loc
  FROM dept
 ORDER BY dept_id
"""

# 실행 
cursor.execute(sql)

# 결과출력
for row in cursor:
    print(row)

# 종료
cursor.close()
conn.close()

print('Oracle 접속 종료.')

Oracle 연결 성공! 21.3.0.0.0
(10, '개발부', 'BUSAN')
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
(40, '인사부', 'BUSAN')
(100, '생산부', 'KWANGJU')
Oracle 접속 종료.


#### INSERT 실행

- DBEAVER에서 TEST 스키마에 SEQ_DEPT 시퀀스 생성

In [44]:
# 전체 SELECT 단위 코드

import oracledb

# DB접속정보 설정
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user = 'test', password = 'test12345', dsn = dsn)
print('Oracle 연결 성공!', conn.version)

try: 
    # 커서 생성
    cursor = conn.cursor()

    # INSERT 쿼리 작성
    sql = """INSERT INTO dept(dept_id, dept_name, loc)
            VALUES (SEQ_DEPT.NEXTVAL, :1, :2) """

    # INSERT 쿼리실행
    dept_name = '생산부'; loc = 'KWANGJU'
    cursor.execute(sql, (dept_name, loc))
    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생', e.args)
    conn.rollback()
finally:
    conn.close()

print('INSERT 완료')

Oracle 연결 성공! 21.3.0.0.0
예외발생 (<oracledb.errors._Error object at 0x000001D8A08311F0>,)
INSERT 완료


#### WHERE 조건 실행

In [45]:
import oracledb

# DB접속정보 설정
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user = 'test', password = 'test12345', dsn = dsn)
print('Oracle 연결 성공!', conn.version)

# 커서 생성
cursor = conn.cursor()

# SQL문 실행
sql = """
SELECT dept_id, dept_name, loc
  FROM dept
 WHERE loc = :loc
 ORDER BY dept_id
"""

# 실행, 조건절 파라미터 추가
cursor.execute(sql, loc = 'SEOUL')
rows = cursor.fetchall() # 데이터양이 작을때 한번에 모두 가져오기

# 결과출력
for row in rows:
    print(row)

# 종료
cursor.close()
conn.close()

print('Oracle 접속 종료.')

Oracle 연결 성공! 21.3.0.0.0
(20, '데이터부', 'SEOUL')
(30, '영업부', 'SEOUL')
Oracle 접속 종료.


#### UPDATE 실행

In [46]:
import oracledb

# DB접속정보 설정
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user = 'test', password = 'test12345', dsn = dsn)
print('Oracle 연결 성공!', conn.version)

try:
    # 커서 생성
    cursor = conn.cursor()

    # UPDATE 쿼리 작성
    sql = """UPDATE dept
             SET dept_name = :1
               , loc = :2
             WHERE dept_id = :3"""

    # UPDATE 쿼리실행
    dept_name = '자재부'
    loc = 'INCHEON'
    dept_id = 50
    
    cursor.execute(sql, (dept_name, loc, dept_id))
    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생', e.args)
    conn.rollback()
finally:
    conn.close()

print('UPDATE 완료')

Oracle 연결 성공! 21.3.0.0.0
UPDATE 완료


#### DELETE 실행

In [47]:
import oracledb

# DB접속정보 설정
dsn = 'localhost:1521/XE'
conn = oracledb.connect(user = 'test', password = 'test12345', dsn = dsn)
print('Oracle 연결 성공!', conn.version)

try:
    # 커서 생성
    cursor = conn.cursor()

    # DELETE 쿼리 작성
    sql = """DELETE FROM dept
             WHERE dept_id = :1"""
    dept_id = 50
    # 쿼리 실행
    cursor.execute(sql, (dept_id,))
    print('삭제 행 : ', cursor.rowcount)
    # 트랜잭션 처리
    conn.commit()

    cursor.close()
except Exception as e:
    print('예외발생', e.args)
    conn.rollback()
finally:
    conn.close()

print('DELETE 완료')

Oracle 연결 성공! 21.3.0.0.0
삭제 행 :  0
DELETE 완료
